# Targeting Individuals: Does Personalization Work?

**PSAM 3707 / UN 3707: Persuasion at Scale**
Columbia University, Spring 2026

This notebook accompanies Lecture 21 (April 15). We simulate personality-based ad targeting (Matz et al.), measure the diminishing returns to microtargeting (Tappin et al.), and test whether LLM-generated personalized messages beat generic ones (Simchon et al.).

**Papers:**
- Matz, Kosinski, Nave & Stillwell (2017). "Psychological Targeting as Digital Mass Persuasion." *PNAS*.
- Tappin, Wittenberg, Hewitt, Berinsky & Rand (2023). "Quantifying the Potential Persuasive Returns to Political Microtargeting." *PNAS*.
- Simchon, Halpern et al. (2024). "Persuasive Effects of Political Microtargeting in the Age of Generative AI." *PNAS Nexus*.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Persuasion-at-Scale/targeting/blob/main/notebooks/targeting.ipynb)

In [ ]:
!pip install -q numpy matplotlib scipy pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 13,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## Part 1: The Cambridge Analytica Simulation (Matz et al. 2017)

Cambridge Analytica claimed they could match ads to people's personalities. Matz et al. tested this claim in a real experiment on 3.5 million Facebook users.

The idea: people high in **openness** respond to novelty-focused ads ("Try something new!"), while people low in openness respond to familiarity-focused ads ("Stick with what you love!").

Let's simulate this. We create a population with Big Five personality traits and test whether personality-matched ads actually outperform mismatched ones.

In [ ]:
# Simulate 10,000 people with Big Five personality traits
# Each trait is on a 0-1 scale (normalized)

N = 10000

# Generate correlated personality traits (Big Five)
# Openness, Conscientiousness, Extraversion, Agreeableness, Neuroticism
personality = np.random.beta(2, 2, size=(N, 5))
trait_names = ['Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Neuroticism']

# Classify people as "high openness" or "low openness" (median split, like Matz et al.)
openness = personality[:, 0]
high_openness = openness > np.median(openness)

# Two ad versions:
# - "Novelty ad": appeals to high-openness people
# - "Familiarity ad": appeals to low-openness people

# Base click rate for everyone: 2% (typical for Facebook ads)
base_rate = 0.02

# Personality-matching bonus: +40% click rate when matched (Matz et al. finding)
match_bonus = 0.40

# Simulate: each person sees either the matched or mismatched ad
def run_targeting_experiment(matched=True):
    """Run the Matz-style experiment. If matched=True, high-openness people see the
    novelty ad and low-openness people see the familiarity ad. If matched=False, reversed."""
    clicks = np.zeros(N)
    for i in range(N):
        if matched:
            # Correct match: high openness -> novelty, low openness -> familiarity
            rate = base_rate * (1 + match_bonus)
        else:
            # Mismatch: high openness -> familiarity, low openness -> novelty
            rate = base_rate
        clicks[i] = np.random.random() < rate
    return clicks

# Run both conditions
np.random.seed(42)
matched_clicks = run_targeting_experiment(matched=True)
mismatched_clicks = run_targeting_experiment(matched=False)

print("Matz et al. Replication (Simulated)")
print("=" * 40)
print(f"Matched ads:     {matched_clicks.mean():.3%} click rate")
print(f"Mismatched ads:  {mismatched_clicks.mean():.3%} click rate")
print(f"Improvement:     {(matched_clicks.mean() - mismatched_clicks.mean()) / mismatched_clicks.mean():.0%}")
print(f"\nMatz et al. found: +40% clicks, +50% purchases")

**The Eckles critique:** Dean Eckles (MIT) pointed out that Facebook's ad platform *already* optimizes for engagement. If personality-matched ads are also engagement-matched, then "personality targeting" may just be letting Facebook's algorithm do what it already does for free. The personality part may add nothing beyond what the platform already knows.

## Part 2: Diminishing Returns to Targeting (Tappin et al. 2023)

Tappin et al. asked: if you had *perfect* targeting data, how much better could you do than just sending everyone the best single message?

Their key finding: knowing someone's **party ID** captures almost all the benefit. Adding personality traits, demographics, or psychographic profiles adds nearly nothing.

Let's simulate this by progressively adding targeting variables and measuring the marginal gain of each one.

In [ ]:
# Simulate a population with multiple targeting variables
# and measure how much each additional variable helps

N = 5000
n_messages = 6  # different political messages to choose from

# Generate targeting variables for each person
np.random.seed(42)
party = np.random.choice([0, 1], size=N)          # 0 = Dem, 1 = Rep
age_group = np.random.choice([0, 1, 2], size=N)   # young, middle, old
gender = np.random.choice([0, 1], size=N)          # 0 = M, 1 = F
education = np.random.choice([0, 1], size=N)       # 0 = no college, 1 = college
openness_score = np.random.beta(2, 2, size=N)      # Big Five openness (continuous)

# True persuasion effect of each message depends on covariates
# Party is the dominant predictor (reflects Tappin et al. finding)
# Other variables add progressively smaller contributions

def true_effect(person_idx, message):
    """True persuasion effect of a message on a person.
    Party determines ~80% of the variation. Other variables add noise."""
    p = party[person_idx]
    a = age_group[person_idx]
    g = gender[person_idx]
    e = education[person_idx]
    o = openness_score[person_idx]

    # Base effect depends heavily on party-message match
    # Messages 0-2 work better on Dems, messages 3-5 work better on Reps
    if message < 3:
        party_effect = 5.0 * (1 - p) - 2.0 * p  # good for Dems, bad for Reps
    else:
        party_effect = 5.0 * p - 2.0 * (1 - p)  # good for Reps, bad for Dems

    # Age adds a small interaction (message 1 works better on young people)
    age_effect = 0.3 * (1 if (message == 1 and a == 0) else 0)

    # Gender adds even less
    gender_effect = 0.1 * (1 if (message == 2 and g == 1) else 0)

    # Education adds even less
    edu_effect = 0.05 * (1 if (message == 4 and e == 1) else 0)

    # Openness adds nearly nothing (Cambridge Analytica's supposed edge)
    openness_effect = 0.02 * o * (1 if message == 0 else -0.5)

    # Add noise
    noise = np.random.normal(0, 1)

    return party_effect + age_effect + gender_effect + edu_effect + openness_effect + noise


# For each person, compute the best message under different targeting strategies
def best_message_uniform():
    """No targeting: pick the single message that works best on average."""
    avg_effects = np.zeros(n_messages)
    for m in range(n_messages):
        effects = [true_effect(i, m) for i in range(N)]
        avg_effects[m] = np.mean(effects)
    return np.argmax(avg_effects)

def best_message_by_group(group_labels):
    """Target by group: pick the best message for each group."""
    unique_groups = np.unique(group_labels)
    best = np.zeros(N, dtype=int)
    for g in unique_groups:
        mask = group_labels == g
        indices = np.where(mask)[0]
        avg_effects = np.zeros(n_messages)
        for m in range(n_messages):
            effects = [true_effect(i, m) for i in indices]
            avg_effects[m] = np.mean(effects)
        best[mask] = np.argmax(avg_effects)
    return best

# Compute average persuasion under each targeting strategy
np.random.seed(42)

# Strategy 0: No targeting (uniform best message)
uniform_msg = best_message_uniform()
uniform_effects = [true_effect(i, uniform_msg) for i in range(N)]

# Strategy 1: Target by party only
np.random.seed(42)
party_msgs = best_message_by_group(party)
party_effects = [true_effect(i, party_msgs[i]) for i in range(N)]

# Strategy 2: Target by party + age
np.random.seed(42)
party_age = party * 10 + age_group  # combined group label
party_age_msgs = best_message_by_group(party_age)
party_age_effects = [true_effect(i, party_age_msgs[i]) for i in range(N)]

# Strategy 3: Target by party + age + gender
np.random.seed(42)
party_age_gender = party * 100 + age_group * 10 + gender
pag_msgs = best_message_by_group(party_age_gender)
pag_effects = [true_effect(i, pag_msgs[i]) for i in range(N)]

# Strategy 4: Target by party + age + gender + education
np.random.seed(42)
full_demo = party * 1000 + age_group * 100 + gender * 10 + education
full_msgs = best_message_by_group(full_demo)
full_effects = [true_effect(i, full_msgs[i]) for i in range(N)]

strategies = ['Uniform\n(no targeting)', 'Party\nonly', 'Party +\nAge', 'Party + Age\n+ Gender',
              'Party + Age +\nGender + Edu']
means = [np.mean(uniform_effects), np.mean(party_effects), np.mean(party_age_effects),
         np.mean(pag_effects), np.mean(full_effects)]

print("Average persuasion effect by targeting strategy:")
for s, m in zip(strategies, means):
    print(f"  {s.replace(chr(10), ' '):30s}: {m:.2f} pp")

In [ ]:
# Figure 1: Diminishing returns to targeting (Tappin et al. replication)
fig, ax = plt.subplots(figsize=(10, 6))

colors_bar = ['#95a5a6', '#e74c3c', '#e67e22', '#f1c40f', '#2ecc71']
bars = ax.bar(range(len(strategies)), means, color=colors_bar, edgecolor='white', width=0.7)

# Add value labels on bars
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')

ax.set_xticks(range(len(strategies)))
ax.set_xticklabels(strategies, fontsize=11)
ax.set_ylabel('Average Persuasion Effect (pp)', fontsize=13)
ax.set_title('Diminishing Returns to Microtargeting\n(Tappin et al. 2023 pattern)', fontsize=14)

# Annotate the big jump vs the small jumps
ax.annotate('Big jump:\nknowing party',
            xy=(0.5, (means[0] + means[1])/2), xytext=(1.5, means[0] - 0.5),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2),
            fontsize=11, color='#e74c3c', fontweight='bold')

ax.annotate('Diminishing returns:\nadding more variables',
            xy=(3, means[3]), xytext=(3.5, means[1] + 0.3),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5),
            fontsize=10, color='gray')

plt.tight_layout()
plt.show()

**What you should see:** A large jump from "no targeting" to "party only," then rapidly diminishing returns as you add more variables. Knowing someone's party affiliation captures most of the benefit. Adding age, gender, education, and personality on top of party adds almost nothing.

This is Tappin et al.'s central finding: **the Cambridge Analytica pitch was a $15 million solution to a problem that a free variable (party registration, which is public) already solves.**

## Part 3: Simchon et al. (2024) - Does GPT-4 Make Targeting Work?

Maybe the problem was that human-written ads couldn't exploit personality differences effectively. What if GPT-4 wrote a *unique* message for each person? Would targeting finally deliver?

Simchon et al. found: GPT-4 is genuinely persuasive (up to 12pp attitude shifts), but **targeted messages are no more persuasive than generic ones.**

In [ ]:
# Simchon et al. (2024) simulation
# Three conditions: control (no message), generic GPT-4 message, targeted GPT-4 message
# Key finding: targeting adds nothing over generic

np.random.seed(42)
n_per_condition = 2800  # ~8,600 total like the paper

# Simulate attitude shifts (percentage points)
# Control: no shift (mean 0, sd 5)
control_shifts = np.random.normal(0, 5, n_per_condition)

# Generic message: persuasive (mean 6.2pp, matching paper)
generic_shifts = np.random.normal(6.2, 5, n_per_condition)

# Targeted message: slightly lower, not significantly different (mean 4.8pp, matching paper)
targeted_shifts = np.random.normal(4.83, 5, n_per_condition)

# Figure 2: Compare the three conditions
fig, ax = plt.subplots(figsize=(9, 6))

conditions = ['Control\n(no message)', 'Generic\nGPT-4 message', 'Targeted\nGPT-4 message']
shift_data = [control_shifts, generic_shifts, targeted_shifts]
colors_cond = ['#95a5a6', '#3498db', '#e74c3c']

# Violin plot
parts = ax.violinplot(shift_data, positions=[0, 1, 2], showmeans=True, showmedians=False)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors_cond[i])
    pc.set_alpha(0.7)
parts['cmeans'].set_color('black')
parts['cmeans'].set_linewidth(2)

# Add means as text
for i, (data, name) in enumerate(zip(shift_data, conditions)):
    ax.text(i, np.mean(data) + 1.5, f'{np.mean(data):.1f}pp',
            ha='center', fontsize=13, fontweight='bold')

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(conditions, fontsize=12)
ax.set_ylabel('Attitude Shift (percentage points)', fontsize=13)
ax.set_title('Simchon et al. (2024): GPT-4 Is Persuasive, But Targeting Adds Nothing', fontsize=14)
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)

# Add significance annotation
ax.annotate('n.s. (p = 0.226)', xy=(1.5, max(np.mean(generic_shifts), np.mean(targeted_shifts)) + 2),
            fontsize=12, ha='center', color='gray',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#f8f9fa', edgecolor='gray'))

plt.tight_layout()
plt.show()

**What you should see:** Both GPT-4 conditions shift attitudes relative to control (the violin distributions are shifted right). But the generic and targeted conditions overlap almost completely. The difference between them is not statistically significant (p = 0.226 in the actual paper).

The punchline: **the most sophisticated language model on earth, with full targeting information, cannot beat a well-crafted generic argument.** Argument quality drives persuasion, not argument-person matching.

## Part 4: Alignment and Misalignment

The same targeting technology can help or harm, depending on whose interests it serves.

| Application | Aligned with user? | Example |
|-------------|-------------------|---------|
| Netflix thumbnails | Yes | You find shows you enjoy faster |
| Amazon recommendations | Yes | You discover useful products |
| Cambridge Analytica | No | Changing your political views without your knowledge |
| Facebook emotional contagion | No | Studying you without consent |

Let's visualize the alignment spectrum by simulating user satisfaction under different targeting objectives.

In [ ]:
# Figure 3: Alignment spectrum
# Simulate user outcomes under different targeting objectives

np.random.seed(42)
n_users = 1000

# User's true preference score (what they actually want)
user_preference = np.random.normal(0, 1, n_users)

# Scenario 1: Aligned targeting (Netflix-style: optimize for user satisfaction)
# The algorithm learns what you like and shows it to you
aligned_outcome = user_preference + np.random.normal(2, 0.5, n_users)  # shifted positive

# Scenario 2: Misaligned targeting (Cambridge Analytica-style: optimize for persuasion)
# The algorithm targets your vulnerabilities
misaligned_outcome = -np.abs(user_preference) + np.random.normal(-1, 1, n_users)  # shifted negative

# Scenario 3: No targeting (random content)
random_outcome = np.random.normal(0, 1.5, n_users)

fig, ax = plt.subplots(figsize=(10, 6))

data = [random_outcome, aligned_outcome, misaligned_outcome]
labels = ['No Targeting\n(random content)', 'Aligned Targeting\n(Netflix: maximize\nuser satisfaction)',
          'Misaligned Targeting\n(Cambridge Analytica:\nmaximize persuasion)']
colors_align = ['#95a5a6', '#27ae60', '#c0392b']

bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.6,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors_align):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_ylabel('User Outcome\n(positive = user benefits, negative = user harmed)', fontsize=12)
ax.set_title('Same Technology, Different Objectives, Different Outcomes', fontsize=14)

plt.tight_layout()
plt.show()

**What you should see:** Aligned targeting (green) shifts the distribution of user outcomes positive; people find things they genuinely want. Misaligned targeting (red) shifts it negative; the algorithm exploits user vulnerabilities. Same technology, different objectives.

## Summary

| Paper | Claim | Finding |
|-------|-------|---------|
| Matz et al. (2017) | Personality matching works | +40% clicks, but confounded by Facebook's existing optimization |
| Tappin et al. (2023) | Returns to targeting are steep then flat | Party ID captures most of the benefit; personality adds nearly nothing |
| Simchon et al. (2024) | GPT-4 can micro-target | GPT-4 is persuasive, but targeting adds nothing over generic messages |

**The punchline from all three papers:** personalization is less powerful than feared. The Cambridge Analytica fantasy of "pushing your psychological buttons" doesn't work much better than knowing your party affiliation. Even with the most powerful language model available, matching the message to the person doesn't beat a good generic argument.

**The real story was always the infrastructure.** Cambridge Analytica's power was Facebook's API (harvesting 87M profiles from 270K quiz-takers), not their targeting algorithm. The vulnerability was architectural, in the platform's design choices.